## Google Colab

1. Run **Install dependencies** (next cell).
2. Run **Colab paths + Drive** (cell after that). Set `MOUNT_GOOGLE_DRIVE = False` if your data is only under `/content` (then edit `COLAB_DATA_DIR` or upload files there).
3. Put the same Excel/CSV layout as on your Mac in **`MyDrive/Project 005 Data/`** (or change `COLAB_DATA_DIR` in that cell).

On Drive, `read_excel` can hit FUSE errors; this notebook defines **`read_excel_colab_safe`** and uses it for workbook reads where patched.

**If you see `ImportError: cannot import name '_center' from 'numpy._core.umath'`**, Colab's pre-installed numpy was broken by an earlier `pip install -U`. Recover with:

```
!pip install --force-reinstall -q "numpy==2.0.2" "pandas==2.2.2"
```

then **Runtime → Restart session** and run this notebook from the top. The install cell below no longer upgrades numpy/pandas/torch and will stop the notebook if the runtime needs a restart.


In [ ]:
# Install dependencies.
# Colab (esp. GPU runtime w/ RAPIDS) pins numpy<2.1 and pandas<2.4. Any `pip install -U`
# can desync wheels and break `numpy._core.umath._center` (imported by numpy.strings,
# xgboost, sklearn, etc.). We therefore:
#   1) Detect a broken/split numpy,
#   2) Delete shadow dist dirs pip left behind (`~numpy*`, `~orch*`, half-removed numpy),
#   3) Surgically reinstall numpy==2.0.2 and pandas==2.2.2 (compatible with RAPIDS),
#   4) Halt the notebook with a clear "Restart session" message.
# We never use `-U` on Colab and never touch torch here; the GPU runtime ships its own.
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

_IN_COLAB = "google.colab" in sys.modules
_USE_TORCH = False
_USE_STATS = True


def _pip(*args):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        check=True,
    )


def _has(module: str) -> bool:
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False


def _numpy_healthy() -> bool:
    # If numpy._core.umath._center is missing, the numpy install is split across
    # incompatible wheels and Python cannot recover in the current session.
    try:
        from numpy._core import umath as _um
        import numpy.strings  # noqa: F401  same path xgboost/sklearn trigger
        return hasattr(_um, "_center")
    except Exception:
        return False


def _colab_cleanup_and_repair_numpy() -> None:
    # Wipe pip's abandoned shadow installs (names start with '~') and any half-removed
    # numpy dir/dist-info — these are what make `pip install --force-reinstall numpy`
    # silently fail to fully overwrite the old files.
    import glob
    site_roots = [Path(p) for p in sys.path if p.endswith("dist-packages") or p.endswith("site-packages")]
    site_roots = [p for p in site_roots if p.is_dir()]
    for root in site_roots:
        for pat in ("~*", "numpy", "numpy-*.dist-info", "numpy.libs"):
            for hit in glob.glob(str(root / pat)):
                try:
                    if Path(hit).is_dir():
                        shutil.rmtree(hit, ignore_errors=True)
                    else:
                        Path(hit).unlink(missing_ok=True)
                except Exception:
                    pass
    # Reinstall a numpy/pandas pair that's compatible with Colab GPU (RAPIDS: numpy<2.1).
    _pip("--no-deps", "--force-reinstall", "numpy==2.0.2")
    _pip("--force-reinstall", "pandas==2.2.2")


_COLAB_RESTART_MSG = (
    "Colab runtime was repaired (numpy/pandas reinstalled to versions compatible with "
    "RAPIDS / GPU runtime). Please **Runtime -> Restart session** and then run this "
    "notebook from the top."
)

if _IN_COLAB:
    if not _numpy_healthy():
        _colab_cleanup_and_repair_numpy()
        raise RuntimeError(_COLAB_RESTART_MSG)
    _needed = []
    for _mod, _pkg in [
        ("xgboost", "xgboost"),
        ("openpyxl", "openpyxl"),
        ("tqdm", "tqdm"),
        ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"),
    ]:
        if not _has(_mod):
            _needed.append(_pkg)
    if _USE_TORCH and not _has("torch"):
        _needed.append("torch")
    if _USE_STATS and not _has("statsmodels"):
        _needed.append("statsmodels")
    if _needed:
        _pip(*_needed)  # intentionally NO -U
        raise RuntimeError(
            "Installed missing packages on Colab: " + ", ".join(_needed) + ". "
            "Please **Runtime -> Restart session**, then run the notebook from the top."
        )
    print("Colab env OK - no pip actions needed.")
else:
    _PKGS = ["pandas", "numpy", "matplotlib", "openpyxl", "scikit-learn", "tqdm", "xgboost"]
    if _USE_TORCH:
        _PKGS = ["torch", *_PKGS]
    if _USE_STATS:
        _PKGS.append("statsmodels")
    _pip("-U", *_PKGS)
    print("pip OK | Colab: False")


In [ ]:
# --- Colab: paths, optional Drive mount, safe Excel reads ---
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
MOUNT_GOOGLE_DRIVE = True  # False if you only use /content uploads
COLAB_DATA_DIR = Path("/content/drive/MyDrive/Project 005 Data")


def _marker_exists(p: Path, name: str = "Master_Data_AVG_version_update.xlsx") -> bool:
    try:
        return (p / name).is_file()
    except OSError:
        return False


def resolve_project005_base_dir() -> Path:
    if IN_COLAB:
        if COLAB_DATA_DIR.is_dir() and _marker_exists(COLAB_DATA_DIR):
            return COLAB_DATA_DIR.resolve()
        alt = Path("/content")
        if _marker_exists(alt):
            return alt.resolve()
        return COLAB_DATA_DIR.resolve()
    here = Path.cwd().resolve()
    if _marker_exists(here):
        return here
    fb = (Path.home() / "Downloads" / "Project 005 Data").resolve()
    if fb.is_dir() and _marker_exists(fb):
        return fb
    return here


if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped or failed:", exc)

BASE_DIR = resolve_project005_base_dir()
if IN_COLAB and BASE_DIR.is_dir():
    try:
        os.chdir(BASE_DIR)
    except OSError:
        pass


def read_excel_colab_safe(path, **kwargs):
    """Read Excel; on Colab copy from Drive to /tmp first (reduces FUSE transport errors)."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    resolved = path.expanduser().resolve()
    sp = os.fspath(resolved)
    if IN_COLAB and sp.startswith("/content/drive/"):
        local = Path("/tmp") / f"_colab_safe_{resolved.name}"
        shutil.copyfile(resolved, local)
        try:
            return pd.read_excel(local, **kwargs)
        finally:
            try:
                local.unlink(missing_ok=True)
            except OSError:
                pass
    return pd.read_excel(resolved, **kwargs)


print("IN_COLAB:", IN_COLAB, "| BASE_DIR:", BASE_DIR)


In [ ]:
# --- Colab NumPy guard (before heavy imports): numpy._core.umath must expose _center. ---
import sys as _sys
if "google.colab" in _sys.modules:
    try:
        from numpy._core import umath as _np_um
        import numpy.strings  # noqa: F401  same path xgboost/sklearn trigger
        _np_ok = hasattr(_np_um, "_center")
    except Exception:
        _np_ok = False
    if not _np_ok:
        import glob as _glob, shutil as _shutil, subprocess as _sp
        from pathlib import Path as _Path
        _roots = [_Path(p) for p in _sys.path if p.endswith("dist-packages") or p.endswith("site-packages")]
        for _root in [_r for _r in _roots if _r.is_dir()]:
            for _pat in ("~*", "numpy", "numpy-*.dist-info", "numpy.libs"):
                for _hit in _glob.glob(str(_root / _pat)):
                    try:
                        if _Path(_hit).is_dir():
                            _shutil.rmtree(_hit, ignore_errors=True)
                        else:
                            _Path(_hit).unlink(missing_ok=True)
                    except Exception:
                        pass
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", "numpy==2.0.2"], check=True)
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "pandas==2.2.2"], check=True)
        raise RuntimeError(
            "Colab numpy was broken and has been reinstalled (2.0.2). "
            "Please **Runtime -> Restart session**, then run the notebook from the top."
        )



In [40]:
import pandas as pd


C5

In [41]:
file_name = "/content/aiColumbia MSBA C5 Project Data 2026 012726.xlsx"
navios_xl = read_excel_colab_safe(file_name, sheet_name = None)

df = navios_xl["C5 Rate"][2:navios_xl["C5 Rate"][navios_xl["C5 Rate"].isnull().all(axis=1)].index.tolist()[0]]
df.columns = df.iloc[0]
df = df[1:]
df.reset_index(drop=True, inplace=True)

df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace = True)
df.columns.name = None
df.columns = ["C5"]
df


FileNotFoundError: [Errno 2] No such file or directory: '/content/aiColumbia MSBA C5 Project Data 2026 012726.xlsx'

In [ ]:
sin_month = navios_xl["SIN Monthly"][2:245]

new_columns = [
    f"{str(a).strip()}_{str(b).strip()}".strip('_')
    for a, b in zip(sin_month.iloc[0].fillna(''), sin_month.iloc[1].fillna(''))
]

sin_month.columns = new_columns
sin_month = sin_month.iloc[2:].reset_index(drop=True)
sin_month.reset_index(drop=True, inplace=True)
sin_month["Date"] = pd.to_datetime(sin_month["Date"])
sin_month.set_index("Date", inplace = True)
sin_month.columns.name = None
sin_month


,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes","Japan BFI Production_,000 tonnes","R.O.Korea BFI Production_,000 tonnes",...,"Thermal Coal Price, FOB Australia_$/Tonne",Australia Seaborne Iron Ore Exports_Million Tonnes,China Seaborne Iron Ore Imports_Million Tonnes,China Seaborne Coal Imports_Million Tonnes,China Seaborne Steel Products Exports_Million Tonnes,Capesize Bulkcarrier Deliveries_No,Capesize Bulkcarrier Deliveries_DWT,Capesize Bulkcarrier Fleet Development_No,Capesize Bulkcarrier Fleet Development_DWT million,Capesize Fleet Growth_% Yr/Yr
Date,,,,,,,,,,,,,,,,,,,,,
2006-01-01,52.1,20.1,2.1,6.4,30166,9453,3931,28910,6982,2397,...,43.1875,15.913,26.00701,2.51299,2.21747,8,1447579,657,111.15133,8.682
2006-02-01,52.1,20.1,3.7,20,29462,8878,3740,28334,6430,2149,...,47.7,20.226,24.24644,2.49136,2.23774,4,709004,665,112.59891,8.763
2006-03-01,55.3,17.8,3.1,10,32889,9665,3974,32538,7009,2178,...,49.75,17.986,28.87328,3.5728,3.20631,4,737405,669,113.30791,9.072
2006-04-01,58.1,16.6,3.6,9.5,33711,9356,3870,32451,6656,2057,...,52.875,19.726,26.56734,3.49051,3.2987,7,1252952,673,114.04532,8.468
2006-05-01,54.8,17.9,4.2,11.6,35934,9928,4154,35326,7109,2305,...,52.6,20.015,23.83915,2.90733,4.27761,3,529537,679,115.16839,9.139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-01,49.8,6.5,3.4,11.6,73490,6375.159,4990.276,66050,4612.399,3583.978,...,106.32,82.373,113.58598,35.04683,11.61126,1,210000,2049,405.12171,1.264
2025-10-01,49,4.9,1.6,-8.1,72000,6853.34,5093.145,65550,4850.896,3786.115,...,107.5,84.943,109.05247,32.26956,10.71087,5,998684,2049,405.15875,1.273
2025-11-01,49.2,4.8,-2.1,NaN,69870,6773.827,4965.023,62340,4838.013,3580.672,...,112.6,76.695,108.75787,32.88764,11.24229,4,783971,2054,406.15743,1.414


In [ ]:
sin_week = navios_xl["SIN Weekly"].iloc[3:1042, [0, 6, 7]]
new_columns = [
    f"{str(a).strip()}_{str(b).strip()}".strip('_')
    for a, b in zip(sin_week.iloc[0].fillna(''), sin_week.iloc[1].fillna(''))
]
sin_week.columns = new_columns
sin_week = sin_week.iloc[2:].reset_index(drop=True)
sin_week.reset_index(drop=True, inplace=True)
sin_week["Date"] = pd.to_datetime(sin_week["Date"])
sin_week.set_index("Date", inplace = True)
sin_week.columns.name = None
sin_week


,China 20mm Steel Plate Commodity Price_$/Tonne,"Brent Crude Oil Price, 1 Month Future_$/bbl"
Date,,
2006-01-06,NaN,61.042
2006-01-13,NaN,62.18
2006-01-20,NaN,63.544
2006-01-27,NaN,63.774
2006-02-03,NaN,64
...,...,...
2025-10-17,481.79571,61.994
2025-10-24,479.63382,63.37
2025-10-31,484.83058,65.002


In [ ]:
pip install deep_translator


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.2 MB/s eta 0:00:00


In [ ]:
from deep_translator import GoogleTranslator

iron_prod = read_excel_colab_safe("/content/Iron Ore Production 06-26 month.xlsx")[1:243]
iron_prod.columns = iron_prod.iloc[0]
iron_prod = iron_prod[1:]

translator = GoogleTranslator(source='auto', target='en')

def translate_text(text):
    if pd.isna(text) or text == '':
        return text
    return translator.translate(str(text))

iron_prod['Date'] = iron_prod['Date'].apply(translate_text)
iron_prod



1,Date,"Iron Ore Production (10,000 tonnes)","Cumulated Iron Ore Production (10,000 Tonnes)",Iron Ore Production Increase YoY (%),Cumulated Iron Ore Production YoY (%)
2,January 2026,NaN,NaN,NaN,NaN
3,December 2025,7934.5,98371.5,-4.4,-2.8
4,November 2025,8302.8,92362.2,3.7,-2.8
5,October 2025,8403.3,85173.6,-2.9,-3.2
6,September 2025,8426.7,76142.9,0.6,-3.8
...,...,...,...,...,...
238,May 2006,4611.9,19124,38,31.5
239,April 2006,4049.6,14471.3,22.3,29.4
240,March 2006,4556.1,10294.2,42.4,32.8
241,February 2006,2947.6,5466.3,36.8,26.4


In [ ]:
iron_prod["Date"] = pd.to_datetime(iron_prod["Date"])
iron_prod.set_index("Date", inplace = True)
iron_prod.columns.name = None
iron_prod


/tmp/ipython-input-400/49288200.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iron_prod["Date"] = pd.to_datetime(iron_prod["Date"])


,"Iron Ore Production (10,000 tonnes)","Cumulated Iron Ore Production (10,000 Tonnes)",Iron Ore Production Increase YoY (%),Cumulated Iron Ore Production YoY (%)
Date,,,,
2026-01-01,NaN,NaN,NaN,NaN
2025-12-01,7934.5,98371.5,-4.4,-2.8
2025-11-01,8302.8,92362.2,3.7,-2.8
2025-10-01,8403.3,85173.6,-2.9,-3.2
2025-09-01,8426.7,76142.9,0.6,-3.8
...,...,...,...,...
2006-05-01,4611.9,19124,38,31.5
2006-04-01,4049.6,14471.3,22.3,29.4
2006-03-01,4556.1,10294.2,42.4,32.8


In [ ]:
lpi = read_excel_colab_safe("/content/LPI Data.xlsx").iloc[:,:2]
lpi["Date"] = pd.to_datetime(lpi["Date"])
lpi.reset_index(drop=True, inplace=True)
lpi.set_index("Date", inplace = True)
lpi


,Value
Date,
2026-01-01,51.2
2025-12-01,50.8
2025-11-01,50.9
2025-10-01,50.7
2025-09-01,51.2
...,...
2012-03-01,59.5
2012-02-01,55.8
2012-01-01,53.3


In [ ]:
steel_prod = read_excel_colab_safe("/content/index_iron1.xlsx")
steel_prod["Date"] = pd.to_datetime(steel_prod["Date"])
steel_prod.reset_index(drop=True, inplace=True)
steel_prod.set_index("Date", inplace = True)
steel_prod


,"Crude Steel Production, Current Period (10,000 Tons)","Crude Steel Production, Year-to-Date (10,000 Tons)","Crude Steel Production, Year-on-Year Growth (%)","Crude Steel Production, Year-to-Date (YTD) Growth (%)","Steel Products Production, Current Period (10,000 Tons)","Steel Products Production, Year-to-Date (10,000 Tons)","Steel Products Production, Year-on-Year Growth (%)","Steel Products Production, Year-to-Date (YTD) Growth (%)","Rebar Production, Current Period (10,000 Tons)","Rebar Production, Year-to-Date (10,000 Tons)","Rebar Production, Year-on-Year Growth (%)","Rebar Production, Year-to-Date (YTD) Growth (%)","Pig Iron Production, Current Period (10,000 Tons)","Pig Iron Production, Year-to-Date (10,000 Tons)","Pig Iron Production, Year-on-Year Growth (%)","Pig Iron Production, Year-to-Date (YTD) Growth (%)"
Date,,,,,,,,,,,,,,,,
2025-12-01,6817.7,96081.2,-10.3,-4.4,11531.0,144612.1,-3.8,3.1,1355.9,18630.8,-15.6,-4.3,6072.2,83604.1,-9.9,-3.0
2025-11-01,6987.1,89166.5,-10.9,-4.0,11591.0,133277.0,-2.6,4.0,1375.1,17295.3,-17.6,-3.2,6234.3,77404.6,-8.7,-2.3
2025-10-01,7199.7,81787.4,-12.1,-3.9,11863.8,121759.2,-0.9,4.7,1434.0,15801.0,-18.6,-2.0,6554.9,71137.3,-7.9,-1.8
2025-09-01,7349.0,74624.9,-4.6,-2.9,12420.9,110384.7,5.1,5.4,1475.0,14338.7,-2.9,-0.1,6604.6,64586.2,-2.4,-1.1
2025-08-01,7736.9,67180.6,-0.7,-2.8,12276.5,98217.1,9.7,5.5,1541.2,12867.8,23.6,0.3,6979.3,57907.0,1.0,-1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006-05-01,3593.4,16268.5,19.6,18.6,4023.3,18059.8,27.1,24.6,690.6,3233.6,23.9,22.0,3532.6,15818.3,23.0,21.1
2006-04-01,3371.1,12624.1,19.0,18.0,3831.1,14030.7,27.5,24.2,646.9,2543.9,22.0,21.6,3245.1,12278.5,17.4,20.1
2006-03-01,3288.9,9219.0,20.1,17.6,3800.4,10202.4,21.8,21.7,660.2,1897.3,16.6,21.6,3253.8,8995.5,22.2,20.7


In [ ]:
bloom_names = pd.ExcelFile("/content/DATA12346789 (2).xlsx").sheet_names
bloom_names


['M2 china money supply',
 'china new loan',
 '70cities-houseprices',
 'chn_ironore_stockpile',
 'chn_steel_price',
 'copper',
 'DXY',
 'EURUSD',
 'CNYUSD',
 'AUDUSD',
 'LPR1Y_CHN',
 'IRONORE_FUT',
 'CCFI',
 'pmba_vlsfo_0.5',
 'BDTI']

In [ ]:

bloomberg = read_excel_colab_safe("/content/DATA12346789 (2).xlsx", sheet_name=None)

processed_dfs = []

for name, df in bloomberg.items():
    df = df.set_index("Date")
    df.columns = [name]
    processed_dfs.append(df)

df_final = pd.concat(processed_dfs, axis=1)

df_final


,M2 china money supply,china new loan,70cities-houseprices,chn_ironore_stockpile,chn_steel_price,copper,DXY,EURUSD,CNYUSD,AUDUSD,LPR1Y_CHN,IRONORE_FUT,CCFI,pmba_vlsfo_0.5,BDTI
Date,,,,,,,,,,,,,,,
2006-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.1821,0.12391,0.7332,NaN,NaN,NaN,NaN,NaN
2006-01-03,NaN,NaN,NaN,NaN,NaN,4440.0,89.840,1.2019,0.12391,0.7403,NaN,NaN,NaN,NaN,1893.0
2006-01-04,NaN,NaN,NaN,NaN,NaN,4540.0,89.140,1.2119,0.12391,0.7469,NaN,NaN,NaN,NaN,1849.0
2006-01-05,NaN,NaN,NaN,NaN,NaN,4512.0,89.330,1.2110,0.12395,0.7475,NaN,NaN,NaN,NaN,1825.0
2006-01-06,NaN,NaN,NaN,NaN,NaN,4514.0,88.910,1.2151,0.12396,0.7541,NaN,NaN,NaN,NaN,1797.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-04,NaN,NaN,NaN,NaN,2260.0,13044.5,97.616,1.1807,0.14401,0.6998,NaN,102.17,NaN,NaN,1678.0
2026-02-05,NaN,NaN,NaN,NaN,2260.0,12903.0,97.824,1.1777,0.14413,0.6927,NaN,101.03,NaN,NaN,1679.0
2026-02-06,NaN,NaN,NaN,16001.0,2260.0,12994.0,97.633,1.1815,0.14419,0.7013,NaN,100.11,1122.15,NaN,1691.0


In [ ]:
list_of_dfs = [df, sin_month, sin_week, iron_prod, lpi, steel_prod, df_final]

df_master = pd.concat(list_of_dfs, axis=1)

df_master.sort_index(inplace=True)


In [ ]:
df_master


,C5,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes","Japan BFI Production_,000 tonnes",...,copper,DXY,EURUSD,CNYUSD,AUDUSD,LPR1Y_CHN,IRONORE_FUT,CCFI,pmba_vlsfo_0.5,BDTI
Date,,,,,,,,,,,,,,,,,,,,,
2006-01-01,NaN,52.1,20.1,2.1,6.4,30166,9453,3931,28910,6982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.1821,0.12391,0.7332,NaN,NaN,NaN,NaN,NaN
2006-01-03,8.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4440.0,89.840,1.2019,0.12391,0.7403,NaN,NaN,NaN,NaN,1893.0
2006-01-04,9.175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4540.0,89.140,1.2119,0.12391,0.7469,NaN,NaN,NaN,NaN,1849.0
2006-01-05,9.355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4512.0,89.330,1.2110,0.12395,0.7475,NaN,NaN,NaN,NaN,1825.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13044.5,97.616,1.1807,0.14401,0.6998,NaN,102.17,NaN,NaN,1678.0
2026-02-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12903.0,97.824,1.1777,0.14413,0.6927,NaN,101.03,NaN,NaN,1679.0
2026-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12994.0,97.633,1.1815,0.14419,0.7013,NaN,100.11,1122.15,NaN,1691.0


In [ ]:
file_path = '/content/drive/MyDrive/Project 005 Data/Master_Data_Base_version.xlsx'
df_master.to_excel(file_path, index=True)


In [ ]:
file_path = '/content/drive/MyDrive/Project 005 Data/Master_Data_Base_version.xlsx'
df_master = read_excel_colab_safe(file_path)


In [ ]:
bloom_names = pd.ExcelFile("Data 2.xlsx").sheet_names

processed_dfs = []

for name in bloom_names:
    df = read_excel_colab_safe("Data 2.xlsx", sheet_name=name).iloc[:, :2]
    df = df.set_index("Date")
    df.columns = [name]
    processed_dfs.append(df)


In [ ]:
bloom2_df = pd.concat(processed_dfs, axis=1)
bloom2_df


,Coal Price week,Brent,MSCI World Energy Index,Wheat Price,Aluminum 3 month future,S&P 500,Hang Seng,Euro Stoxx 50,MSCI Global,MSCI EM,KOSPI,VIX,VHSI,VSTOXX,Nikkei
Date,,,,,,,,,,,,,,,
2005-12-09,NaN,57.31,NaN,293.00,2266.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-12,NaN,59.44,NaN,299.50,2231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-13,NaN,59.52,NaN,304.00,2249.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-14,NaN,59.60,NaN,309.50,2220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-15,NaN,59.85,NaN,318.75,2220.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-09,695.0,69.04,313.1932,528.75,3125.5,6964.82,27027.16,6059.01,4570.79,1539.54,5298.04,17.36,23.02,17.9193,56363.94
2026-02-10,NaN,68.80,312.8272,528.25,3093.0,6941.81,27183.15,6047.06,4570.51,1549.73,5301.69,17.79,22.03,18.0028,57650.54
2026-02-11,NaN,69.40,320.3092,537.25,3103.0,6941.47,27266.38,6035.64,4568.94,1564.48,5354.49,17.65,21.38,18.0027,NaN


In [ ]:
trade = pd.ExcelFile("Tradeflow - Australia & Brazil Exports (Daily, dated back to 2013).xlsx").sheet_names


In [ ]:
x = read_excel_colab_safe("Tradeflow - Australia & Brazil Exports (Daily, dated back to 2013).xlsx", sheet_name= trade[0]).T.reset_index(drop = True).T
x.columns = x.iloc[0]
x = x.iloc[1:4694]


,Voy Intake (MT)
Date,
2013-01-01,1523019
2013-01-02,665618
2013-01-03,672353
2013-01-04,1071859
2013-01-05,487532
...,...
2025-12-26,1191206
2025-12-27,1101297
2025-12-28,1032077


In [ ]:
processed_dfs1 = []
for name in trade:
  try:
    x = read_excel_colab_safe("Tradeflow - Australia & Brazil Exports (Daily, dated back to 2013).xlsx", sheet_name= name).T.reset_index(drop = True).T
    x.columns = x.iloc[0]
    x = x[1:sum(x["Load Date"] != "Grand total")]
    x["Date"] = pd.to_datetime(x["Load Date"], format = "%d/%m/%Y")
    x.drop(columns = ["Load Date"], inplace = True)
    x.reset_index(drop=True, inplace=True)

    x.set_index("Date", inplace = True)

    x.columns = [name + " (MT)"]
    processed_dfs1.append(x)

  except:
    continue

  print(name)

trade_df = pd.concat(processed_dfs1, axis=1)
trade_df


AUS-CHN Iron Ore
AUS-CHN Coal
AUS-CHN Bauxite
AUS-JPN Iron Ore
AUS-S Korea Iron Ore
AUS-Taiwan Iron Ore
Brazil-CHN Iron Ore
Brazil-S Korea Iron Ore
Brazil-Taiwan Iron Ore


,AUS-CHN Iron Ore (MT),AUS-CHN Coal (MT),AUS-CHN Bauxite (MT),AUS-JPN Iron Ore (MT),AUS-S Korea Iron Ore (MT),AUS-Taiwan Iron Ore (MT),Brazil-CHN Iron Ore (MT),Brazil-S Korea Iron Ore (MT),Brazil-Taiwan Iron Ore (MT)
Date,,,,,,,,,
2013-01-01,1523019,157834,NaN,385194,NaN,314822,662390,NaN,255271
2013-01-02,665618,141391,NaN,160423,NaN,NaN,NaN,NaN,NaN
2013-01-03,672353,284207,NaN,316640,188209,NaN,158036,NaN,NaN
2013-01-04,1071859,NaN,NaN,172107,198405,NaN,NaN,NaN,NaN
2013-01-05,487532,NaN,NaN,336743,196249,NaN,171287,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2025-12-27,1101297,NaN,NaN,NaN,397348,NaN,539765,NaN,NaN
2025-12-28,1032077,293628,NaN,NaN,171539,NaN,377755,NaN,NaN
2025-12-29,542301,254077,NaN,NaN,NaN,NaN,394448,NaN,NaN


In [ ]:
import_export = read_excel_colab_safe("Iron Ore & Coal Imports & Exports from Clarksons (Monthly, dated back to 2010).xlsx")[3:]
import_export
new_columns = [
    f"{str(a).strip()}_{str(b).strip()}".strip('_')
    for a, b in zip(import_export.iloc[0].fillna(''), import_export.iloc[1].fillna(''))
]
import_export.columns = new_columns
import_export = import_export.iloc[2:].reset_index(drop=True)
import_export.reset_index(drop=True, inplace=True)
import_export = import_export[1:sum(~(import_export.iloc[:,1].isnull()))]
import_export["Date"] = pd.to_datetime(import_export["Date"])
import_export.set_index("Date", inplace = True)
import_export.columns.name = None
import_export


,Australia Seaborne Iron Ore Exports_Million Tonnes,Brazil Seaborne Iron Ore Exports_Million Tonnes,China Seaborne Iron Ore Imports_Million Tonnes,South Korea Seaborne Iron Ore Imports_Million Tonnes,Japan Seaborne Iron Ore Imports_Million Tonnes,Taiwan Seaborne Iron Ore Imports_Million Tonnes
Date,,,,,,
2010-02-01,29.667,23.422,48.44757,3.723,9.691,1.166
2010-03-01,32.166,27.351,57.70583,5.211,11.869,1.231
2010-04-01,32.717,21.363,53.94097,4.631,11.592,1.255
2010-05-01,32.274,23.437,50.48049,4.17,11.023,1.616
2010-06-01,34.659,24.45,45.76271,4.929,11.376,1.216
...,...,...,...,...,...,...
2025-08-01,75.083,40.074,103.01535,5.318,7.877,1.656
2025-09-01,82.373,36.415,113.58598,5.5528,7.844,1.508
2025-10-01,84.943,42.42,109.05247,6.805,8.383,1.852


In [ ]:
bunker = read_excel_colab_safe("Platts Snigapore Bunkerwire (Daily).xlsx").iloc[4:, [0, 1, 5, 9]]
new_columns = [
    f"{str(a).strip()}_{str(b).strip()}".strip('_')
    for a, b in zip(bunker.iloc[0].fillna(''), bunker.iloc[1].fillna(''))
]
bunker.columns = new_columns
bunker = bunker[2:]
bunker.reset_index(inplace = True, drop = True)
bunker["Date"] = pd.to_datetime(bunker["Date"])
bunker.set_index("Date", inplace = True)
bunker


,HFO_PX_LAST,MGO_PX_LAST,VLSFO_PX_LAST
Date,,,
2026-02-12,442,679,486
2026-02-11,436,679.75,489
2026-02-10,432,668,485
2026-02-09,423,669.75,480
2026-02-06,430,670,487
...,...,...,...
2010-01-07,508,NaN,NaN
2010-01-06,502.5,NaN,NaN
2010-01-05,500,NaN,NaN


In [ ]:
c5_ffa = read_excel_colab_safe("C5 FFA.xlsx")
c5_ffa


,GroupDesc,ArchiveDate,RouteIdentifier,RouteAverage,FFADescription
0,BFA Cape,2010-01-04,C5+1MON,13.250,2026-02-10
1,BFA Cape,2010-01-04,C5+2MON,12.875,2026-03-10
2,BFA Cape,2010-01-05,C5+1MON,13.125,2026-02-10
3,BFA Cape,2010-01-05,C5+2MON,12.688,2026-03-10
4,BFA Cape,2010-01-06,C5+1MON,12.725,2026-02-10
...,...,...,...,...,...
12183,BFA Cape,2026-02-10,C5+2MON,10.183,2026-04-26
12184,BFA Cape,2026-02-10,C5+3MON,10.289,2026-05-26
12185,BFA Cape,2026-02-11,C5+1MON,10.700,2026-03-26
12186,BFA Cape,2026-02-11,C5+2MON,10.710,2026-04-26


In [ ]:
split_df = {ident: group_df for ident, group_df in c5_ffa.groupby("RouteIdentifier")}
mon1_ffa = split_df["C5+1MON"].loc[:,["ArchiveDate", "RouteAverage"]]
mon1_ffa.columns = ["Date", "1 Month FFA Route Average"]
mon1_ffa["Date"] = pd.to_datetime(mon1_ffa["Date"])
mon1_ffa.reset_index(drop = True, inplace = True)
mon1_ffa.set_index("Date", inplace = True)
mon1_ffa.columns.name = None
mon1_ffa


,1 Month FFA Route Average
Date,
2010-01-04,13.250
2010-01-05,13.125
2010-01-06,12.725
2010-01-07,11.613
2010-01-08,11.450
...,...
2026-02-05,10.280
2026-02-06,10.255
2026-02-09,10.253


In [ ]:
mon2_ffa = split_df["C5+2MON"].loc[:,["ArchiveDate", "RouteAverage"]]
mon2_ffa.columns = ["Date", "2 Month FFA Route Average"]
mon2_ffa["Date"] = pd.to_datetime(mon2_ffa["Date"])
mon2_ffa.reset_index(drop = True, inplace = True)
mon2_ffa.set_index("Date", inplace = True)
mon2_ffa.columns.name = None
mon2_ffa


,2 Month FFA Route Average
Date,
2010-01-04,12.875
2010-01-05,12.688
2010-01-06,12.225
2010-01-07,11.238
2010-01-08,11.125
...,...
2026-02-05,10.118
2026-02-06,10.142
2026-02-09,10.152


In [ ]:
mon3_ffa = split_df["C5+3MON"].loc[:,["ArchiveDate", "RouteAverage"]]
mon3_ffa.columns = ["Date", "3 Month FFA Route Average"]
mon3_ffa["Date"] = pd.to_datetime(mon3_ffa["Date"])
mon3_ffa.reset_index(drop = True, inplace = True)
mon3_ffa.set_index("Date", inplace = True)
mon3_ffa.columns.name = None
mon3_ffa


,3 Month FFA Route Average
Date,
2010-01-29,9.925
2010-02-01,9.310
2010-02-02,9.325
2010-02-03,9.310
2010-02-04,9.480
...,...
2026-02-05,10.226
2026-02-06,10.216
2026-02-09,10.247


In [ ]:
bunker_week = read_excel_colab_safe("Clarksons Singapore Bunkerwire (Weekly).xlsx").iloc[3:]
new_columns = [
    f"{str(a).strip()}_{str(b).strip()}, Weekly Data".strip('_')
    for a, b in zip(bunker_week.iloc[0].fillna(''), bunker_week.iloc[1].fillna(''))
]
bunker_week.columns = new_columns
bunker_week = bunker_week[2:sum(~(bunker_week.iloc[:,1].isnull()))]
bunker_week = bunker_week.rename(columns = {"Date, Weekly Data": "Date"})
bunker_week.reset_index(drop=True, inplace=True)
bunker_week["Date"] = pd.to_datetime(bunker_week["Date"])
bunker_week.set_index("Date", inplace = True)
bunker_week


,"HSFO 380cst Bunker Prices (3.5% Sulphur), Singapore_$/Tonne, Weekly Data","MGO Bunker Prices, Singapore_$/Tonne, Weekly Data","VLSFO Bunker Prices (0.5% Sulphur), Singapore_$/Tonne, Weekly Data"
Date,,,
2010-01-01,481.5,632.5,NaN
2010-01-08,508,657.5,NaN
2010-01-15,490,642.5,NaN
2010-01-22,479.5,627.5,NaN
2010-01-29,462,602.5,NaN
...,...,...,...
2026-01-09,357.75,594.25,423.25
2026-01-16,374,624,443.5
2026-01-23,388.5,634.75,456


In [ ]:
df_master.reset_index(inplace = True, drop = True)
df_master["Date"] = pd.to_datetime(df_master["Date"])
df_master.set_index("Date", inplace = True)


In [ ]:
df_master_update


,C5,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes","Japan BFI Production_,000 tonnes",...,Taiwan Seaborne Iron Ore Imports_Million Tonnes,HFO_PX_LAST,MGO_PX_LAST,VLSFO_PX_LAST,"HSFO 380cst Bunker Prices (3.5% Sulphur), Singapore_$/Tonne, Weekly Data","MGO Bunker Prices, Singapore_$/Tonne, Weekly Data","VLSFO Bunker Prices (0.5% Sulphur), Singapore_$/Tonne, Weekly Data",1 Month FFA Route Average,2 Month FFA Route Average,3 Month FFA Route Average
Date,,,,,,,,,,,,,,,,,,,,,
2005-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,423,669.75,480,NaN,NaN,NaN,10.253,10.152,10.247
2026-02-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,432,668,485,NaN,NaN,NaN,10.208,10.183,10.289
2026-02-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,436,679.75,489,NaN,NaN,NaN,10.700,10.710,10.683


In [85]:
df = read_excel_colab_safe("Master_Data_Base_version_updated.xlsx")
df.head()


,Date,C5,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes",...,Taiwan Seaborne Iron Ore Imports_Million Tonnes,HFO_PX_LAST,MGO_PX_LAST,VLSFO_PX_LAST,"HSFO 380cst Bunker Prices (3.5% Sulphur), Singapore_$/Tonne, Weekly Data","MGO Bunker Prices, Singapore_$/Tonne, Weekly Data","VLSFO Bunker Prices (0.5% Sulphur), Singapore_$/Tonne, Weekly Data",1 Month FFA Route Average,2 Month FFA Route Average,3 Month FFA Route Average
0,2005-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-12-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-12-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-12-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2005-12-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
steel_price = read_excel_colab_safe("analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm", sheet_name="daily steel price", header=1)


In [52]:
# Set the first row as columns
steel_price.columns = steel_price.iloc[0]

# Drop the row that was used for headers
steel_price = steel_price[1:]

# Reset index and fix the Date column name if needed
steel_price.reset_index(drop=True, inplace=True)

# Rename the first column to 'Date' if it contains dates
steel_price.rename(columns={steel_price.columns[0]: 'Date'}, inplace=True)

# Filter out rows where Date is NaN and convert to datetime
steel_price = steel_price[steel_price['Date'].notna()]
steel_price['Date'] = pd.to_datetime(steel_price['Date'])

display(steel_price.head())


,Date,Rebar 12mm,Rebar 20mm,Wire Rod 6.5mm high speed,Wire Rod 6.5 mm,HRC 3.0mm,HRC 4.75mm,CRC 0.5mm,CRC 1.0mm,Medium Plate Low-Alloy 20mm,...,Tangshan Billet 20MnSi,NaN,NaN,NaN,NaN,HRC premium (discount) to rebar,NaN,NaN,NaN,NaN
0,2010-01-04,3290.598291,3128.205128,3350.42735,3230.769231,3401.709402,3333.333333,4641.025641,4982.905983,3547.008547,...,3008.547009,NaN,2010.0,1.0,NaN,111.111111,NaN,NaN,NaN,NaN
1,2010-01-05,3307.692308,3145.299145,3367.521368,3247.863248,3418.803419,3341.880342,4666.666667,5000,3572.649573,...,3025.641026,NaN,2010.0,1.0,NaN,111.111111,NaN,NaN,NaN,NaN
2,2010-01-06,3307.692308,3145.299145,3384.615385,3273.504274,3418.803419,3341.880342,4666.666667,5000,3572.649573,...,3051.282051,NaN,2010.0,1.0,NaN,111.111111,NaN,NaN,NaN,NaN
3,2010-01-07,3333.333333,3162.393162,3418.803419,3307.692308,3452.991453,3341.880342,4700.854701,5000,3615.384615,...,3094.017094,NaN,2010.0,1.0,NaN,119.65812,NaN,NaN,NaN,NaN
4,2010-01-11,3290.598291,3205.128205,3401.709402,3256.410256,3418.803419,3333.333333,4700.854701,5000,3615.384615,...,3094.017094,NaN,2010.0,1.0,NaN,128.205128,NaN,NaN,NaN,NaN


In [53]:
steel_price.reset_index(drop = True, inplace = True)
steel_price.set_index("Date", inplace = True)
steel_price.columns.name = None


In [114]:
steel_price = steel_price.iloc[:, :23]


In [103]:
# Read with first two rows as multi-level column names; use US$/t date column as index
hrc_price = read_excel_colab_safe(
    "analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm",
    sheet_name="global HRC price",
    header=[0, 1]
)
# Date column is under US$/t (second column, index 1) - contains YYYY-MM-DD format
date_col = hrc_price.columns[1]
hrc_price = hrc_price.set_index(date_col)
hrc_price.index = pd.to_datetime(hrc_price.index)
# Drop the year column (first column under US$/t)
hrc_price = hrc_price.drop(columns=hrc_price.columns[0], errors="ignore")
# Flatten MultiIndex columns to string: e.g. "USD/ST_US"
hrc_price.columns = ["_".join(str(c) for c in col).strip("_") if isinstance(col, tuple) else str(col) for col in hrc_price.columns]
# Rename index (US$/t) to Date, add hrc_ prefix to all column names
hrc_price.index.name = "Date"
hrc_price.columns = ["hrc_" + str(col) for col in hrc_price.columns]


In [104]:
hrc_price.columns.name = None


In [105]:
hrc_price = hrc_price.iloc[:756, list(range(6)) + list(range(8, 13))]
hrc_price


,hrc_USD/ST_US,hrc_USD/CWT_EU,hrc_JPY/Ton_Japan,hrc_USD/ST_CIS,hrc_USD/MT_China,hrc_USD/MT_SEA,hrc_USD/MT_US,hrc_USD/MT_EU,hrc_USD/MT_Japan,hrc_USD/MT_CIS,hrc_USD/MT_China.1
Date,,,,,,,,,,,
2010-01-08,545,24.72,55000.0,467.21,NaN,570.0,600.882029,486.614173,593.567883,515.115766,NaN
2010-01-15,560,25.45,55000.0,471.74,NaN,575.0,617.420066,500.984252,605.927068,520.110254,NaN
2010-01-22,580,25.04,55000.0,471.74,NaN,580.0,639.470783,492.913386,612.335783,520.110254,NaN
2010-01-27,580,26.13,55000.0,471.74,NaN,570.0,639.470783,514.370079,611.043217,520.110254,NaN
2010-02-05,590,25.41,58000.0,471.74,NaN,570.0,650.496141,500.196850,649.859944,520.110254,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2024-11-08,700,27.02,92000.0,485.35,512.0,493.0,771.775083,531.889764,602.725367,535.115766,512.0
2024-11-15,700,26.80,92000.0,476.28,508.0,490.0,771.775083,527.559055,596.241089,525.115766,508.0
2024-11-22,690,26.18,92000.0,476.28,503.0,490.0,760.749724,515.354331,594.392040,525.115766,503.0


In [107]:
scrap_price = read_excel_colab_safe("analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm", sheet_name="daily iron ore and scrap price", header = None).iloc[1:, :]


In [108]:

# Reset column names to values from row 1 and set first column name to 'Date'
scrap_price.columns = scrap_price.iloc[0]
scrap_price = scrap_price.iloc[1:].reset_index(drop=True)
scrap_price.rename(columns={scrap_price.columns[0]: 'Date'}, inplace=True)
scrap_price.head()



1,Date,MB spot iron ore price (US$/t),Shanghai rebar price (Rmb/t),NaN,Turkey scrap price (US$/t),NaN,"Jiangyin scrap price (Rmb/t, ex-VAT)",Processing Cost,EAF Margin Spot vs. Spot(RHS),EAF Margin 1 week scrap inventory(RHS),"Graphite electrode (Rmb/t, ex-VAT)",Cash cost (Rmb/t),EAF cash margin (Rmb/t),NaN,MB spot iron ore price WoW
0,2011-01-03 00:00:00,167.13,4008.547009,NaN,458,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-01-04 00:00:00,167.46,4008.547009,NaN,480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-01-05 00:00:00,169.48,4017.094017,NaN,480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011-01-06 00:00:00,169.05,4034.188034,NaN,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011-01-07 00:00:00,171.43,4034.188034,NaN,515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [109]:
scrap_price.reset_index(drop = True, inplace = True)
scrap_price.set_index("Date", inplace = True)
scrap_price.columns.name = None



/opt/anaconda3/lib/python3.13/site-packages/pandas/core/indexes/base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


In [110]:
scrap_price = scrap_price.iloc[:3824, [1, 3, 5, 6, 7, 8, 9, 10, 11]]
scrap_price


,Shanghai rebar price (Rmb/t),Turkey scrap price (US$/t),"Jiangyin scrap price (Rmb/t, ex-VAT)",Processing Cost,EAF Margin Spot vs. Spot(RHS),EAF Margin 1 week scrap inventory(RHS),"Graphite electrode (Rmb/t, ex-VAT)",Cash cost (Rmb/t),EAF cash margin (Rmb/t)
Date,,,,,,,,,
2011-01-03,4008.547009,458,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-01-04,4008.547009,480,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-01-05,4017.094017,480,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-01-06,4034.188034,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-01-07,4034.188034,515,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-02-06,2840.707965,338,2180,1153.846154,-493.138189,-463.138189,13716.814159,3151.318584,-310.610619
2026-02-09,2840.707965,338,2180,1153.846154,-493.138189,-463.138189,13716.814159,3151.318584,-310.610619
2026-02-10,2840.707965,338,2180,1153.846154,-493.138189,-493.138189,13716.814159,3151.318584,-310.610619


In [65]:
trade_inv = read_excel_colab_safe("analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm", sheet_name="steel inventory", header=None).iloc[:, :9]
trade_inv.head()


,0,1,2,3,4,5,6,7,8
0,NaN,Traders' inventory(mt),NaN,NaN,NaN,NaN,NaN,NaN,10.2667
1,NaN,NaN,Rebar,Wire rod,HRC,Medium plate,CRC,NaN,Total
2,2006-04-21 00:00:00,2006-04-21 00:00:00,2.87847,0.78837,1.45477,0.7357,1.17865,NaN,7.03596
3,2006-04-28 00:00:00,2006-04-28 00:00:00,2.77779,0.69422,1.39073,0.71165,1.18384,NaN,6.75823
4,2006-05-12 00:00:00,2006-05-12 00:00:00,2.6849,0.59105,1.32885,0.70287,1.15143,NaN,6.4591


In [127]:
trade_inv = read_excel_colab_safe("analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm", sheet_name="steel inventory", header=None).iloc[:1020, :7]

# Column names with "trader inv " prefix for all columns
col_names = ["trader inv " + (str(x).strip() if pd.notna(x) and str(x).strip() else f"col{i}") for i, x in enumerate(trade_inv.iloc[1])]
col_names[0] = "Date"  # First column is Date
trade_inv.columns = col_names
trade_inv = trade_inv.iloc[2:].reset_index(drop=True)  # drop header rows, keep up to 1018 data rows

# Convert Date to datetime (handles both Excel serial numbers and datetime strings)
trade_inv["Date"] = pd.to_datetime(trade_inv["Date"], errors='coerce')
trade_inv = trade_inv[trade_inv["Date"].notna()].reset_index(drop=True)

# Remove trader inv col1 column and set Date as index
trade_inv = trade_inv.drop(columns=["trader inv col1"], errors='ignore')
trade_inv = trade_inv.set_index("Date")
trade_inv


,trader inv Rebar,trader inv Wire rod,trader inv HRC,trader inv Medium plate,trader inv CRC
Date,,,,,
2006-04-21,2.87847,0.78837,1.45477,0.7357,1.17865
2006-04-28,2.77779,0.69422,1.39073,0.71165,1.18384
2006-05-12,2.6849,0.59105,1.32885,0.70287,1.15143
2006-05-19,2.66147,0.56051,1.32524,0.70359,1.12324
2006-05-26,2.52805,0.5084,1.28279,0.6645,1.06316
...,...,...,...,...,...
2026-01-15,2.9541,0.5014,2.858,1.2132,1.1366
2026-01-22,3.0312,0.4945,2.8114,1.2115,1.136
2026-01-29,3.264,0.5028,2.7833,1.1971,1.1601


In [117]:
trade_inv_consum = read_excel_colab_safe("analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm", sheet_name="steel apparent consumption", header=None).iloc[:1020, :7]

# Column names with "trader inv " prefix for all columns
col_names = ["trader inv consumption " + (str(x).strip() if pd.notna(x) and str(x).strip() else f"col{i}") for i, x in enumerate(trade_inv_consum.iloc[1])]
col_names[0] = "Date"  # First column is Date
trade_inv_consum.columns = col_names
trade_inv_consum = trade_inv_consum.iloc[2:].reset_index(drop=True)  # drop header rows, keep up to 1018 data rows

# Convert Date to datetime (handles both Excel serial numbers and datetime strings)
trade_inv_consum["Date"] = pd.to_datetime(trade_inv_consum["Date"], errors='coerce')
trade_inv_consum = trade_inv_consum[trade_inv_consum["Date"].notna()].reset_index(drop=True)

# Remove trader inv col1 column and set Date as index
trade_inv_consum = trade_inv_consum.drop(columns=["trader inv consumption col1"], errors='ignore')
trade_inv_consum = trade_inv_consum.set_index("Date")
trade_inv_consum


,trader inv consumption Rebar,trader inv consumption Wire rod,trader inv consumption HRC,trader inv consumption Medium plate,trader inv consumption CRC
Date,,,,,
2006-04-21,2.87847,0.78837,1.45477,0.7357,1.17865
2006-04-28,2.77779,0.69422,1.39073,0.71165,1.18384
2006-05-12,2.6849,0.59105,1.32885,0.70287,1.15143
2006-05-19,2.66147,0.56051,1.32524,0.70359,1.12324
2006-05-26,2.52805,0.5084,1.28279,0.6645,1.06316
...,...,...,...,...,...
2026-01-15,2.9541,0.5014,2.858,1.2132,1.1366
2026-01-22,3.0312,0.4945,2.8114,1.2115,1.136
2026-01-29,3.264,0.5028,2.7833,1.1971,1.1601


In [68]:
import pandas as pd

# Read ore inventory with raw structure (A:I)
raw = read_excel_colab_safe(
    "analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm",
    sheet_name="ore inventory",
    header=None,
    usecols="A:I",
).iloc[:1020]

# Row 3 has headers, row 4 has units; combine both for robust naming
h1 = raw.iloc[2].ffill()
h2 = raw.iloc[3]
col_names = ["Date"]
for i in range(1, raw.shape[1]):
    top = str(h1.iloc[i]).strip() if pd.notna(h1.iloc[i]) else ""
    sub = str(h2.iloc[i]).strip() if pd.notna(h2.iloc[i]) else ""
    name = top if top else sub
    if not name:
        name = f"metric{i}"
    col_names.append(f"ore inv {name}")

ore_inv = raw.copy()

# Make column names unique to avoid duplicate-label selection issues
unique_cols = []
seen = {}
for c in col_names:
    n = seen.get(c, 0)
    unique_cols.append(c if n == 0 else f"{c} ({n})")
    seen[c] = n + 1

ore_inv.columns = unique_cols
ore_inv = ore_inv.iloc[4:].reset_index(drop=True)  # data starts at row 5

# Parse Date and keep valid rows
ore_inv["Date"] = pd.to_datetime(ore_inv["Date"], errors="coerce")
ore_inv = ore_inv[ore_inv["Date"].notna()].reset_index(drop=True)

# Drop duplicate date-like column (same values as Date)
drop_col = None
for i in range(1, ore_inv.shape[1]):
    s = ore_inv.iloc[:, i]
    dt = pd.to_datetime(s, errors="coerce")
    valid = dt.notna()
    if valid.any() and (dt[valid].eq(ore_inv.loc[valid, "Date"]).mean() > 0.95):
        drop_col = ore_inv.columns[i]
        break

if drop_col is not None:
    ore_inv = ore_inv.drop(columns=[drop_col])

ore_inv = ore_inv.set_index("Date")
ore_inv


,ore inv Ore inventory at ports,ore inv From Australia,ore inv From Brazil,ore inv Average port inventory,ore inv Iron ore days at steel mills - RHS,ore inv Average iron ore days at steel mills - RHS,ore inv Average iron ore days at steel mills - RHS (1)
Date,,,,,,,
2008-08-15,73.42,NaN,NaN,NaN,NaN,NaN,NaN
2008-08-22,71.99,NaN,NaN,NaN,NaN,NaN,NaN
2008-08-29,74.46,NaN,NaN,NaN,NaN,NaN,NaN
2008-09-05,75.46,NaN,NaN,NaN,NaN,NaN,NaN
2008-09-12,74.92,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2026-01-16,165.506,NaN,NaN,119.312467,22,25.642177,0.01671
2026-01-23,167.628,NaN,NaN,119.312467,24,25.642177,0.012821
2026-01-30,170.1804,NaN,NaN,119.312467,29,25.642177,0.015227


In [69]:
# Coking coal price (based on sheet layout shown in screenshot)
file_path = "analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm"
raw = read_excel_colab_safe(file_path, sheet_name="coking coal price", header=None)

# Header is split across 3 rows (name / unit / basis)
# Detect header row by looking for "Shanxi Liulin" or "HCC"
header_row = 0
for r in range(min(20, len(raw))):
    row_txt = " | ".join(raw.iloc[r].astype(str).tolist()).lower()
    if ("shanxi liulin" in row_txt) or ("hcc" in row_txt):
        header_row = r
        break

name_row = raw.iloc[header_row]
unit_row = raw.iloc[min(header_row + 1, len(raw) - 1)]
basis_row = raw.iloc[min(header_row + 2, len(raw) - 1)]

# Date is in column B (index 1) per your screenshot; prices start at column C (index 2)
date_col = 1

# Build clean column names from 3 header rows
price_cols = []
for j in range(2, raw.shape[1]):
    a = str(name_row.iloc[j]).strip() if pd.notna(name_row.iloc[j]) else ""
    b = str(unit_row.iloc[j]).strip() if pd.notna(unit_row.iloc[j]) else ""
    c = str(basis_row.iloc[j]).strip() if pd.notna(basis_row.iloc[j]) else ""
    label = " ".join([x for x in [a, b, c] if x and x.lower() != "nan"]).strip()
    if label:
        price_cols.append((j, f"coking coal {label}"))

# Data starts after basis row
data = raw.iloc[header_row + 3:].reset_index(drop=True).copy()

coking_coal_price = pd.DataFrame()
coking_coal_price["Date"] = pd.to_datetime(data.iloc[:, date_col], errors="coerce", dayfirst=True)

for j, col_name in price_cols:
    coking_coal_price[col_name] = pd.to_numeric(data.iloc[:, j], errors="coerce")

# Keep only valid date rows and set index
coking_coal_price = coking_coal_price[coking_coal_price["Date"].notna()].reset_index(drop=True)
coking_coal_price = coking_coal_price.set_index("Date")

coking_coal_price.head()


/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_18025/1847806976.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  coking_coal_price["Date"] = pd.to_datetime(data.iloc[:, date_col], errors="coerce", dayfirst=True)


,"coking coal Shanxi Liulin #4 Rmb/t, with VAT FOR","coking coal Shanxi Liulin #9 Rmb/t, with VAT FOR",coking coal SBB hard coking coal USD/t,coking coal HCC Peak Downs US$/t FOB,coking coal Australia low-vol coking coal (USD/t) - RHS USD/t FOB,coking coal USD/t FOB
Date,,,,,,
2008-04-01,1700.0,1350.0,NaN,NaN,NaN,NaN
2008-04-01,1700.0,1350.0,NaN,NaN,NaN,NaN
2008-04-01,1795.0,1440.0,NaN,NaN,NaN,NaN
2008-04-01,1795.0,1440.0,NaN,NaN,NaN,NaN
2008-07-01,1795.0,1440.0,NaN,NaN,NaN,NaN


In [70]:
# Daily crude steel production (Date + four requested columns)
file_path = "analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm"
raw = read_excel_colab_safe(file_path, sheet_name="Daily crude steel prod", header=None)

# Find header row containing the four target labels
targets = [
    "national daily production",
    "key enterprises daily",
    "non-key enterprises daily",
    "shanghai rebar price",
]
header_row = 0
for r in range(min(25, len(raw))):
    row_txt = " | ".join(raw.iloc[r].astype(str).tolist()).lower()
    if all(t in row_txt for t in targets):
        header_row = r
        break

# Read first 6 columns to handle possible leading blank column
x = raw.iloc[header_row + 1:, :6].reset_index(drop=True).copy()

# Detect whether Date is in col 0 or col 1
cand0 = pd.to_datetime(x.iloc[:, 0], errors="coerce", dayfirst=True)
cand1 = pd.to_datetime(x.iloc[:, 1], errors="coerce", dayfirst=True)
date_col_idx = 0 if cand0.notna().sum() >= cand1.notna().sum() else 1

# Build output columns from detected Date position
if date_col_idx == 0:
    date_series = cand0
    metric_block = x.iloc[:, 1:5].copy()
else:
    date_series = cand1
    metric_block = x.iloc[:, 2:6].copy()

metric_block.columns = [
    "crude steel National daily production (10mt)",
    "crude steel Key enterprises daily (Mtpc)",
    "crude steel Non-key enterprises daily",
    "crude steel Shanghai rebar price (Rmb/t)",
]

daily_crude_steel_prod = metric_block.copy()
daily_crude_steel_prod.insert(0, "Date", date_series)

# Drop non-data rows (e.g., "co", units) and set Date index
daily_crude_steel_prod = daily_crude_steel_prod[daily_crude_steel_prod["Date"].notna()].reset_index(drop=True)
daily_crude_steel_prod = daily_crude_steel_prod.set_index("Date")

daily_crude_steel_prod.head()


/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_18025/1518372.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cand0 = pd.to_datetime(x.iloc[:, 0], errors="coerce", dayfirst=True)


,crude steel National daily production (10mt),crude steel Key enterprises daily (Mtpc),crude steel Non-key enterprises daily,crude steel Shanghai rebar price (Rmb/t)
Date,,,,
2009-01-01,1.323018,1.016609,0.306409,3620
2009-01-11,1.324562,1.043568,0.280994,3620
2009-01-21,1.399551,1.212914,0.186637,3630
2009-02-01,1.4167,1.11616,0.30054,3650
2009-02-11,1.427537,1.124698,0.302839,3690


In [71]:
# Steel margin weekly: keep Date + columns beginning with cash/cast profit
file_path = "analystCitibank China_Steel_China_Steel_and_Raw_Materials_Price_Inventory_Database Steel by Type 022326.xlsm"
raw = read_excel_colab_safe(file_path, sheet_name="Steel margin weekly", header=None)

# Detect header row where cash profit columns appear
header_row = 0
max_hits = -1
for r in range(min(30, len(raw))):
    row_text = raw.iloc[r].astype(str).str.lower()
    hits = row_text.str.contains("cash profit|cast profit", regex=True).sum()
    if hits > max_hits:
        max_hits = hits
        header_row = r

name_row = raw.iloc[header_row]

# Data starts after header row; drop leading non-data rows using date parsing
data = raw.iloc[header_row + 1:].reset_index(drop=True).copy()

# Date is typically in col B (index 1), but detect robustly between first 3 cols
date_candidates = []
for j in range(min(3, data.shape[1])):
    parsed = pd.to_datetime(data.iloc[:, j], errors="coerce", dayfirst=True)
    date_candidates.append((parsed.notna().sum(), j, parsed))
_, date_col_idx, parsed_date = max(date_candidates, key=lambda x: x[0])

steel_margin_cash_profit = pd.DataFrame()
steel_margin_cash_profit["Date"] = parsed_date

# Keep only columns whose header starts with cash/cast profit
for j in range(data.shape[1]):
    h = str(name_row.iloc[j]).strip()
    h_low = h.lower()
    if h_low.startswith("cash profit") or h_low.startswith("cast profit"):
        col_name = "steel margin " + h
        steel_margin_cash_profit[col_name] = pd.to_numeric(data.iloc[:, j], errors="coerce")

# Final cleanup
steel_margin_cash_profit = steel_margin_cash_profit[steel_margin_cash_profit["Date"].notna()].reset_index(drop=True)
steel_margin_cash_profit = steel_margin_cash_profit.set_index("Date")

steel_margin_cash_profit.head()


/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_18025/3508548238.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(data.iloc[:, j], errors="coerce", dayfirst=True)


,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar,steel margin cash profit with cost delayed 1 month for marginal producer of HRC,steel margin cash profit with cost delayed 1 week for marginal producer of HRC,steel margin cash profit with cost delayed 1 month for marginal producer of CRC,steel margin cash profit with cost delayed 1 week for marginal producer of CRC
Date,,,,,,
2006-04-01,NaN,NaN,NaN,NaN,NaN,NaN
2006-07-01,NaN,253.011014,NaN,1450.446911,NaN,-246.988986
2006-07-01,NaN,176.138073,NaN,1238.531235,NaN,-323.861927
2006-07-01,NaN,121.425005,NaN,1094.074578,NaN,-378.574995
2006-07-01,179.50674,188.979462,998.310159,1007.782881,-320.49326,-311.020538


In [72]:
# Guinea Bauxite 2.26: import monthly tables from all sheets, bind on Date
import os
import re
import pandas as pd

file_candidates = [
    "Guinea Bauxite 2.26.xlsx",
    "Guinea Bauxite 2.26.xlsm",
    "guniea bauxite 2.26.xlsx",
    "guniea bauxite 2.26.xlsm",
]
file_path = next((f for f in file_candidates if os.path.exists(f)), None)
if file_path is None:
    raise FileNotFoundError("Guinea Bauxite 2.26 file not found. Set file_path manually.")

sheet_names = [
    "Guinea Total Bauxite Exports",
    "Guinea Bauxite Exports in Capes",
    "CHN Imports from Guinea",
    "CHN Imports from Guinea inCapes",
]
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_to_num = {m: i + 1 for i, m in enumerate(month_order)}
month_set = set(month_order)

def extract_sheet_long(raw, sheet_name):
    row_labels_pos = None
    for r in range(raw.shape[0]):
        for c in range(raw.shape[1]):
            v = raw.iat[r, c]
            if isinstance(v, str) and v.strip().lower() == "row labels":
                row_labels_pos = (r, c)
                break
        if row_labels_pos is not None:
            break
    if row_labels_pos is None:
        return pd.DataFrame()
    hr, hc = row_labels_pos
    year_cols = []
    for c in range(hc + 1, raw.shape[1]):
        v = raw.iat[hr, c]
        if pd.isna(v):
            if year_cols:
                break
            continue
        year = None
        if isinstance(v, (int, float)) and 1900 <= v <= 2030:
            year = str(int(v))
        else:
            s = str(v).strip()
            if s.isdigit() and len(s) == 4:
                year = s
            else:
                dt = pd.to_datetime(v, errors="coerce")
                if pd.notna(dt) and 1900 <= dt.year <= 2030:
                    year = str(int(dt.year))
                else:
                    m = re.search(r"(19|20)\\d{2}", s)
                    if m:
                        year = m.group(0)
        if year:
            year_cols.append((c, year))
    if not year_cols:
        return pd.DataFrame()
    rows = []
    for r in range(hr + 1, raw.shape[0]):
        label = raw.iat[r, hc]
        if pd.isna(label):
            continue
        label = str(label).strip()
        if label.lower() == "grand total":
            break
        if label in month_set:
            row = {"Month": label}
            for c, y in year_cols:
                row[y] = raw.iat[r, c]
            rows.append(row)
    if not rows:
        return pd.DataFrame()
    df_wide = pd.DataFrame(rows)
    df_wide["Month"] = pd.Categorical(df_wide["Month"], categories=month_order, ordered=True)
    df_wide = df_wide.sort_values("Month")
    year_col_names = [y for _, y in year_cols]
    long_rows = []
    for _, r in df_wide.iterrows():
        mo = r["Month"]
        for yr in year_col_names:
            val = r.get(yr)
            if pd.notna(val):
                dt = pd.Timestamp(int(yr), month_to_num[mo], 1)
                long_rows.append({"Date": dt, sheet_name: pd.to_numeric(val, errors="coerce")})
    out = pd.DataFrame(long_rows).dropna(subset=[sheet_name])
    return out.sort_values("Date").reset_index(drop=True)

dfs = []
for sn in sheet_names:
    raw = read_excel_colab_safe(file_path, sheet_name=sn, header=None)
    df = extract_sheet_long(raw, sn)
    if not df.empty:
        dfs.append(df)

if not dfs:
    raise ValueError("No pivot data found in any Guinea Bauxite sheet.")
guinea_bauxite_monthly_imports = dfs[0]
for df in dfs[1:]:
    guinea_bauxite_monthly_imports = guinea_bauxite_monthly_imports.merge(
        df, on="Date", how="outer"
    )
guinea_bauxite_monthly_imports = guinea_bauxite_monthly_imports.sort_values("Date").reset_index(drop=True)

guinea_bauxite_monthly_imports


,Date,Guinea Total Bauxite Exports,Guinea Bauxite Exports in Capes,CHN Imports from Guinea,CHN Imports from Guinea inCapes
0,2013-01-01,1392190.0,NaN,NaN,NaN
1,2013-02-01,1485247.0,NaN,54957.0,NaN
2,2013-03-01,1572983.0,NaN,58773.0,NaN
3,2013-04-01,1613441.0,NaN,176023.0,NaN
4,2013-05-01,1594250.0,NaN,65577.0,NaN
...,...,...,...,...,...
153,2025-10-01,14023214.0,12616355.0,12991006.0,12443778.0
154,2025-11-01,14723914.0,13235255.0,13545255.0,13038451.0
155,2025-12-01,18639777.0,17111607.0,17335343.0,16774771.0
156,2026-01-01,18904565.0,17663664.0,15949311.0,15891886.0


In [73]:
# C3 Historic FFA 1,2 mos: separate into c3 1 mon ffa and c3 2 mon ffa (same pattern as C5)
c3_ffa = read_excel_colab_safe("C3 Historic FFA 1,2 mos.xlsx")
c3_split_df = {ident: group_df for ident, group_df in c3_ffa.groupby("RouteIdentifier")}

c3_1_mon_ffa = c3_split_df["C3+1MON"].loc[:, ["ArchiveDate", "RouteAverage"]]
c3_1_mon_ffa.columns = ["Date", "C3 1 Mon FFA"]
c3_1_mon_ffa["Date"] = pd.to_datetime(c3_1_mon_ffa["Date"])
c3_1_mon_ffa.reset_index(drop=True, inplace=True)
c3_1_mon_ffa.set_index("Date", inplace=True)
c3_1_mon_ffa.columns.name = None

c3_2_mon_ffa = c3_split_df["C3+2MON"].loc[:, ["ArchiveDate", "RouteAverage"]]
c3_2_mon_ffa.columns = ["Date", "C3 2 Mon FFA"]
c3_2_mon_ffa["Date"] = pd.to_datetime(c3_2_mon_ffa["Date"])
c3_2_mon_ffa.reset_index(drop=True, inplace=True)
c3_2_mon_ffa.set_index("Date", inplace=True)
c3_2_mon_ffa.columns.name = None

c3_1_mon_ffa


,C3 1 Mon FFA
Date,
2010-01-04,34.000
2010-01-05,34.250
2010-01-06,32.625
2010-01-07,31.313
2010-01-08,31.157
...,...
2026-03-03,27.240
2026-03-04,28.275
2026-03-05,28.192


In [74]:
c3_2_mon_ffa


,C3 2 Mon FFA
Date,
2010-01-04,33.125
2010-01-05,33.313
2010-01-06,31.900
2010-01-07,30.700
2010-01-08,30.600
...,...
2026-03-03,27.485
2026-03-04,27.702
2026-03-05,27.618


In [128]:
# Concat/merge all datasets into master data update (following Master_Data_Base_version_updated.xlsx flow)
lst_df2 = [
    df,
    c3_1_mon_ffa,
    c3_2_mon_ffa,
    steel_price,
    hrc_price,
    scrap_price,
    trade_inv,
    ore_inv,
    trade_inv_consum,
    daily_crude_steel_prod,
    coking_coal_price,
    steel_margin_cash_profit,
    guinea_bauxite_monthly_imports.set_index("Date"),
]

def to_datetime_index(d):
    """Test if index is datetime; if not, convert to datetime. Returns df with proper datetime index."""
    d = d.copy()
    try:
        if pd.api.types.is_datetime64_any_dtype(d.index):
            d.index = pd.to_datetime(d.index, errors='coerce').normalize()
            return d[d.index.notna()]
    except Exception:
        pass
    try:
        if pd.api.types.is_integer_dtype(d.index) and (len(d.index) == 0 or d.index.max() < 100000):
            for col in ["Date", "date", "Unnamed: 0"]:
                if col in d.columns:
                    d = d.set_index(col)
                    break
            else:
                if len(d.columns) > 0 and pd.api.types.is_datetime64_any_dtype(d.iloc[:, 0]):
                    d = d.set_index(d.columns[0])
        d.index = pd.to_datetime(d.index, errors='coerce').normalize()
        return d[d.index.notna()]
    except Exception:
        d.index = pd.to_datetime(d.index, errors='coerce').normalize()
        return d[d.index.notna()]

# Process df first - it is always the base
df_base = to_datetime_index(df) if df is not None and not df.empty else None
if df_base is not None and not df_base.empty:
    df_base = df_base[~df_base.index.duplicated(keep='first')]

lst_df2_clean = []
for d in lst_df2[1:]:
    if d is not None and not d.empty:
        d = to_datetime_index(d)
        d = d[~d.index.duplicated(keep='first')]
        lst_df2_clean.append(d)

base_idx = df_base.index if df_base is not None and not df_base.empty else lst_df2_clean[0].index if lst_df2_clean else pd.DatetimeIndex([])
df_master_update = (df_base.copy() if df_base is not None and not df_base.empty else lst_df2_clean[0].reindex(base_idx).copy() if lst_df2_clean else pd.DataFrame())
for d in lst_df2_clean:
    df_master_update = df_master_update.join(d, how='left', rsuffix='_dup')
    df_master_update = df_master_update.drop(columns=[c for c in df_master_update.columns if isinstance(c, str) and c.endswith('_dup')], errors='ignore')
df_master_update = df_master_update.sort_index()
df_master_update


,C5,China Manufacturing Purchasing Managers Index (Official)_PMI,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes","Japan BFI Production_,000 tonnes",...,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar,steel margin cash profit with cost delayed 1 month for marginal producer of HRC,steel margin cash profit with cost delayed 1 week for marginal producer of HRC,steel margin cash profit with cost delayed 1 month for marginal producer of CRC,steel margin cash profit with cost delayed 1 week for marginal producer of CRC,Guinea Total Bauxite Exports,Guinea Bauxite Exports in Capes,CHN Imports from Guinea,CHN Imports from Guinea inCapes
Date,,,,,,,,,,,,,,,,,,,,,
2005-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-12-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [129]:
pd.set_option("display.max_rows", 200)
df_master_update.columns[151]


'ore inv Ore inventory at ports'

In [130]:
# Export merged master data to Excel
BASE_DIR = os.fspath(BASE_DIR)  # from Colab setup cells
file_path1 = os.path.join(BASE_DIR, "Master_Data_Base_version_updated.xlsx")
df_master_update.to_excel(file_path1, index=True)
